# 04 — Comparar uma hipótese de desenvolvimento

## Objetivo
Testar o efeito dos pesos de classe mantendo features, algoritmo e seed constantes. Este laboratório não declara uma arquitetura CNN vencedora.

## Entradas
Certificado v2; train/validation. Features idênticas às do NB03.

## Saídas
`reports/experimental_v2/hypotheses/<experiment_id>/`: hipótese, configurações, resultados e conclusão.

## Permissões de dados
Somente train e validation. Não usar calibration ou qualquer conjunto de avaliação posterior.

## Reprodutibilidade
Seed 42. Uma comparação planejada: LogReg balanced versus sem pesos. Sem registry de modelos arbitrários.

Protocolo: [experimental_protocol_v2.md](../docs/experimental_protocol_v2.md).
Os resultados anteriores estão catalogados como `legacy_v1`; não comparar seus scores diretamente aos v2.


In [ ]:
from pathlib import Path
import sys

# Bootstrap de importação; a resolução dos dados fica nos módulos compartilhados.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "scripts/experimental_protocol.py").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from scripts.v2_artifacts import repository_root, read_json, run_id
ROOT = repository_root()

from scripts.v2_data import load_development
from scripts.v2_baselines import run_classical_study, FEATURE_CONFIG, LOGREG_CONFIG
EXECUTE_EXPERIMENTS = False
EXPERIMENT_ID = run_id("class_weight_ablation_v2")
frames, certificate = load_development(ROOT)
display({"hypothesis": "Pesos de classe alteram macro-F1/recall de classes minoritárias.",
         "only_change": "class_weight: balanced versus None", "seed": 42,
         "primary_metric": "validation macro-F1", "split_version": certificate["split_version"]})


## Hipótese
O desbalanceamento pode favorecer NV na função de custo. Alterar pesos pode melhorar recall de classes pequenas, com custo em NV e accuracy.
Esse efeito não é garantido: precisa ser observado com features e seed constantes.

## Configuração e métrica
Usamos os mesmos 54 atributos e a mesma regressão logística do NB03. Apenas `class_weight` muda.
Comparamos macro-F1, recalls por classe, tempo e convergência. Uma única seed determinística não sustenta alegações de estabilidade entre treinamentos CNN.


In [ ]:
experiment = None
if EXECUTE_EXPERIMENTS:
    output = ROOT / "reports/experimental_v2/hypotheses" / EXPERIMENT_ID
    experiment = run_classical_study(ROOT, output, EXPERIMENT_ID, ablation=True)
    display(pd.DataFrame({name: result["validation"]["per_class"]
                          for name, result in experiment["results"].items()}))
    display(pd.DataFrame([{ "config": name, "macro_f1": result["validation"]["macro_f1"],
                            "accuracy": result["validation"]["accuracy"], "seconds": result["fit_seconds"]}
                          for name, result in experiment["results"].items()]))
    recalls = pd.DataFrame({name: {c: values["recall"] for c, values in result["validation"]["per_class"].items()}
                            for name, result in experiment["results"].items()})
    ax = recalls.plot.bar(figsize=(10, 4))
    ax.set(title="Efeito dos pesos — recall por classe em validation", xlabel="Classe", ylabel="Recall")
    ax.set_ylim(0, 1); plt.tight_layout(); plt.show()
    print("Delta de macro-F1:", experiment["macro_f1_delta"])
    print("Conclusão registrada:", experiment["conclusion"])
    print("Artefatos:", output.relative_to(ROOT).as_posix())
else:
    print("Hipótese/configuração preparadas. Ative EXECUTE_EXPERIMENTS para executá-las; não há resultado presumido.")


## Conclusão
A decisão deve considerar o delta e os recalls, inclusive perdas em outras classes. Um resultado positivo justifica investigação delimitada; não demonstra superioridade universal nem autoriza mudar retrospectivamente critérios CNN registrados.

## Artefatos produzidos
Quando habilitado, um diretório imutável guarda design antes do fit, métricas, predições, ambiente e conclusão.

## Próxima etapa
NB05: preservar a receita CNN conhecida, declarar seeds e critérios antes de treinar.
